##Aspect Based Sentiment Classification using DeBERTa-v3-base

In this project, let us try fine-tuning a Transformer-based encoder model - DeBERTa-v3-base - for Aspect Sentiment Classification (ASC) purpose. We are using Kaggle's SemEval-2014 dataset containing laptop reviews and restaurant reviews. For more flavour, we have combined these 2 seperate datasets into one such that the data we will work with has both kinds of reviews. Additionally, we will perform some error-analysis (confusion matrix, per-domain accuracy, aspect-count wise accuracy and saving row-level incorrect predictions for manual tagging) to get a better understanding of the model performance.

With the findings of the error analysis, we will augment the model with a custom weighted loss function to penalise incorrect predictions of the minority class and see if it helps in improving minority recall.






We have chosen DeBERTa-v3-base model for this task due to the following reasons:

*   The first and main reason is it's **Disentangled Attention** feature - which can prove useful for an ABSA usecase like ours. Disentangled attention means that, the model does not only look at a composite representation of content and positional information, but also its isolated representations and several possible combinations of the 2. The higher context is particularly useful when we need to find the aspects within the sentence and the sentiments associated to them.
*   DeBERTa is pretrained using **Replaced Token Detection** mechanism which means, instead of simply trying to predict a masked token, it evaluates if a predicted token makes sense being part of the sentence, taking into account contextual and semantic relationships between tokens. This is known to produce representations particularly strong for classification (our use-case here i.e., Aspect-level Sentiment Classification ASC)

In [ ]:
#Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)


Setting up the data below:

In [ ]:

LAPTOP_CSV = "Laptop_Train.csv"
RESTAURANT_CSV = "Restaurants_Train.csv"

# Mapping file column names to standard names.
# df.columns for each file.
COLUMN_MAP = {
    "Sentence": "sentence",
    "Aspect Term": "aspect",
    "polarity": "polarity",
}

DROP_CONFLICT = True     # we have set this to drop inputs having conflicting sentiments that may not be meaningful for the model to learn from
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 96          # maximum number of tokens per input. 96 should comfortably cover the relatively short sentences that we have in our data
RANDOM_SEED = 42




Next, we load the dataframes with the necessary columns, add a domain column to identify if the review pertains to laptop or restaurant. We then check the shape of the data and the frequency of each sentiment

In [ ]:
#Loading and tagging domain to respective data (laptop, restaurant reviews)
def load_and_tag(path: str, domain: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"[{domain}] raw shape: {df.shape}, columns: {list(df.columns)}")
    df = df.rename(columns=COLUMN_MAP)
    missing = {"sentence", "aspect", "polarity"} - set(df.columns)
    if missing:
        raise ValueError(
            f"[{domain}] missing expected columns after rename: {missing}. "
            f"Update COLUMN_MAP to match this file's real headers."
        )
    df = df[["sentence", "aspect", "polarity"]].copy()
    df["domain"] = domain
    return df


laptop_df = load_and_tag(LAPTOP_CSV, "laptop")
restaurant_df = load_and_tag(RESTAURANT_CSV, "restaurant")


[laptop] raw shape: (2358, 6), columns: ['id', 'Sentence', 'Aspect Term', 'polarity', 'from', 'to']
[restaurant] raw shape: (3693, 6), columns: ['id', 'Sentence', 'Aspect Term', 'polarity', 'from', 'to']


In [ ]:
#concatenating the 2 dataframes so we have a single dataframe containing both laptop and restaurant reviews
df = pd.concat([laptop_df, restaurant_df], ignore_index=True)

#printing out the shape of the data and sentiment-wise count
print(f"\nCombined shape before cleaning: {df.shape}")
print(df["polarity"].value_counts())

df.head()


Combined shape before cleaning: (6051, 4)
polarity
positive    3151
negative    1671
neutral     1093
conflict     136
Name: count, dtype: int64


,sentence,aspect,polarity,domain
0,I charge it at night and skip taking the cord ...,cord,neutral,laptop
1,I charge it at night and skip taking the cord ...,battery life,positive,laptop
2,The tech guy then said the service center does...,service center,negative,laptop
3,The tech guy then said the service center does...,"""sales"" team",negative,laptop
4,The tech guy then said the service center does...,tech guy,neutral,laptop


We find that majority of the reviews are positive, followed by negative and neutral feedback. We also have a few conflicting reviews which for this particular project, we will be eliminating.

Next is some data cleaning

In [ ]:

# Normalizing polarity strings in case of inconsistent casing across the two files.
df["polarity"] = df["polarity"].astype(str).str.strip().str.lower()

# Dropping rows with missing sentence/aspect/polarity
before = len(df)
df = df.dropna(subset=["sentence", "aspect", "polarity"])
print(f"Dropped {before - len(df)} rows with nulls.")

if DROP_CONFLICT:
    before = len(df)
    df = df[df["polarity"] != "conflict"].reset_index(drop=True)
    print(f"Dropped {before - len(df)} 'conflict' rows.")

# Encoding labels
label2id = {label: i for i, label in enumerate(sorted(df["polarity"].unique()))}
id2label = {i: label for label, i in label2id.items()}
df["label"] = df["polarity"].map(label2id)
num_labels = len(label2id)

#viewing the label map, final shape and value counts of polarity across domains
print(f"\nFinal label map: {label2id}")
print(f"Final combined shape: {df.shape}")
print(df.groupby("domain")["polarity"].value_counts())


Dropped 0 rows with nulls.
Dropped 136 'conflict' rows.

Final label map: {'negative': 0, 'neutral': 1, 'positive': 2}
Final combined shape: (5915, 5)
domain      polarity
laptop      positive     987
            negative     866
            neutral      460
restaurant  positive    2164
            negative     805
            neutral      633
Name: count, dtype: int64


We split the data into training (80%), validation (10%) and test (10%) data and save them to csvs for future usage

In [ ]:
#we stratify based on domain+polarity to make sure the split is proportionate across both products and across sentiments. Hence creating a new column for this purpose
df["strat_key"] = df["domain"] + "_" + df["polarity"]

#creating a temporary dataframe (20%) and training dataframe (80%)
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["strat_key"], random_state=RANDOM_SEED
)
#splitting the temporary df into validation and test (10% each)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["strat_key"], random_state=RANDOM_SEED
)

#printing out the length of each dataset
for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}:",len(split_df),"rows")


train: 4732 rows
val: 591 rows
test: 592 rows


In [ ]:
#dropping the strat_key column as we are done with teh stratified split
train_df = train_df.drop(columns="strat_key").reset_index(drop=True)
val_df = val_df.drop(columns="strat_key").reset_index(drop=True)
test_df = test_df.drop(columns="strat_key").reset_index(drop=True)

#saving each dataset to csv files for later usage
train_df.to_csv("absa_train.csv", index=False)
val_df.to_csv("absa_val.csv", index=False)
test_df.to_csv("absa_test.csv", index=False)

Now that we all the required datasets, let's tokenize them

In [ ]:
#Loading DeBERTa's tokenizer (specifically built for this model)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#We define the tokenizing function where provide the tokenizer with a sentence, aspect pair as training examples. Format would look like "[CLS] The battery life is excellent. [SEP] battery life [SEP]""
def tokenize_df(split_df: pd.DataFrame):
    return tokenizer(
        list(split_df["sentence"]),
        list(split_df["aspect"]),
        truncation=True,
        padding="max_length", #so that all examples have 96 tokens for similar tensor shape throughout
        max_length=MAX_LENGTH,
        return_tensors="pt", #The framework we're using here is PyTorch and so we ask the tokenizer to return PyTorch Tensors
    )

#tokenizing all 3 datasets using our function above
train_enc = tokenize_df(train_df)
val_enc = tokenize_df(val_df)
test_enc = tokenize_df(test_df)

# Sanity check: lets decode one example to confirm the [CLS]...[SEP]...[SEP] shape
print("\nSanity check — decoded example 0 from train:")
print(tokenizer.decode(train_enc["input_ids"][0]))




config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            


Sanity check — decoded example 0 from train:
[CLS] Small servings for main entree, i had salmon (wasnt impressed) girlfriend had chicken, it was good.[SEP] salmon[SEP][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD]


Let's now wrap our tokenized data into a format that PyTorch can feed to DeBERTa during training

In [ ]:
#Defining a PyTorch Dataset class called ABSADataset that tells PyTorch the format of accessing 1 example.
#This is of the form {"input_id":..Tokenized values, "attention_mask":..of 0s and 1s to indicate legit vs padded values,"labels":...our labels converted to a PyTorch Tensor}
#Note, encoding contains input_id and attention_mask within itself

class ABSADataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

#Example input:
#{
#    "input_ids": ...,
#    "attention_mask": ...,
#    "labels": ...
#}

#Function to get the length of the dataset (required by PyTorch's Dataset Class)
    def __len__(self):
        return len(self.labels)

#Function to retrieve the idxth (ie number corresponding to index) training example (key value pairs of input_id, attention_mask, labels). For example, if idx = 5, the function gives
#{
#    "input_ids": for the 5th training example,
#    "attention_mask": for the 5th training example,
#    "labels": for the 5th training example
#}
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

#Convert train, validation, test data into pytorch dataset classes using the above
train_dataset = ABSADataset(train_enc, train_df["label"].tolist())
val_dataset = ABSADataset(val_enc, val_df["label"].tolist())
test_dataset = ABSADataset(test_enc, test_df["label"].tolist())

#Printing out the length of each dataset and label mappings
print(f"\nDatasets ready: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
print(f"label2id = {label2id}")




Datasets ready: train=4732, val=591, test=592
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}


Next, configuring our model and setting up the parameters

In [ ]:

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 96
OUTPUT_DIR = "./absa_deberta_v3_base" #we're saving our trained model inside working directory
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)

Loading the pretrained DeBERTa model specifically meant for classification, since we need to output a sentiment class (DeBERTa is mainly a language representation model i.e., produces numeric vectors and so we need to convert those into appropriate sentiments)

Note: We use FP32 for model weights instead of the default FP16 because having FP16 threw errors during a previous run - this is because we encountered a "Attempting to unscale FP16 gradients" error - this happens when some gradients are too small to be accurately represented in FP16, and so PyTorch scales it upward and when it reverse-scales this value, PyTorch's gradient scaler created errors. Hence we have used FP32 (which represents a much wider range of numbers with higher precision but also uses higher memory than FP16 - which works considering the little more complex mechanism of disentangled attention that DeBERTa uses)

In [ ]:
#Further configurations
model = AutoModelForSequenceClassification.from_pretrained(      #This creates the classification architecture specific to our task
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float32,   # <-- forced fp32 weights
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Unexpected items imply that DeBERTa's pretraining checkpoint has old pretraining components that our classification model doesn't need

Missing items imply that due to the classification task we're taking on a slighly deviant task that's not native to DeBERTa, there are parameters initialized that are missing, which we will be filling out using training and fine-tuning



Notice that Pooler and Classifier are the missing ones, which will be the 2 new layers that we will be adding for the sentiment classification task

* Pooler - takes the encoder's full sequence of token-level outputs and condenses them into a single vector representing the whole input (sentence + aspect pair).

* Classifier - a final linear layer that maps that single pooled vector down to logits for each class we use (3 in our case - postive, neutral, negative)

These logits are then passed through the argmax function to provide the index corresponding to the largest logit and this is mapped to its corresponding label of positive/neutral/negative



In [ ]:
#defining evaluation function for the model
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


In [ ]:
#Training arguments and fine-tuning
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,               #saves 2 of most recent checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1", #metric based on which we choose the final model. Because we have a majority class in the data, accuracy can be misleading.
    num_train_epochs=4,
    per_device_train_batch_size=16,   #which makes 296 batches for each epoch
    per_device_eval_batch_size=32,    #higher than train batch size since there is no need to store gradients for evaluation (just forward pass, no backward pass)
    gradient_accumulation_steps=2,    #this means that, we update gradients to weights after 2 batch runs instead of updating at every batch run. This enables us to use a larger batch for gradient updates
                                      #instead of W1 = W0 + g1, W2 = W1 + g2, we do W2 = W0 + g1 + g2
    learning_rate=1e-5,               #common default for DeBERTa-v3 which is more LR-sensitive (a previously used higher learning rate casued the model to unstabilize and give NaN loss)
    weight_decay=0.01,                #regularization method. 0.01 is a universal default for AdamW-based transformer finetuning
    warmup_steps=70,                  #number of steps where learning rate increases to its current assigned value (learning rate warms up during first 59 steps). Generally computed as 10-15% of total training steps (optimizer updates)
                                      #here, 296/2 (dividing because of gradient accumulation) = 148*4 (number of epochs) = 592 training steps. 12% of this is ~70.
    max_grad_norm=1.0,                #gradient clipping to make sure gradients dont explode (values dont get greater than 1). Trainer's default value
    fp16=False,                       #explicitly done since we are using FP32 because of a FP16 gradients unscaling error we met with
    logging_steps=20,                 #logs training loss every 20 training steps
    report_to="none",
    seed=RANDOM_SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)



Note: Through several iterations, it was found that increasing the warmup steps from 10% to 12% slightly improved the model performance - this could be because of the previously mentioned addition of pooler and classifier during training, which the encoder initially is unfamiliar with. Training has to gently teach the added layers of randomness on how to turn encoder representations into sensible classification decisions, while making only small, careful adjustments to the already-good encoder weights underneath

On the other hand, at 15%, (89 steps), model performance started declining. Additionally, smaller learning rates, cosine decay for the learning rate were also experimented with as part of the fine-tuning process.

In [ ]:
#training the model
trainer.train()

print("\nValidation metrics (best checkpoint):")
print(trainer.evaluate())

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.416397,0.588005,0.754653,0.567828
2,1.030865,0.513913,0.813875,0.733351
3,0.816424,0.480383,0.815567,0.772137
4,0.737147,0.481586,0.818951,0.761860


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation metrics (best checkpoint):


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.737147,0.480383,4,0.815567,0.772137


{'eval_loss': 0.4803827106952667, 'eval_accuracy': 0.8155668358714044, 'eval_macro_f1': 0.7721374436049852}


Our 3rd epoch has given us the best macro F1 score at 0.77. Also, we see that with the final epoch, the validation loss is slightly higher and F1 has dropped which could hint a beggining of overfitting but nothing alarming.

Now let us try to do some error analysis on the test set and see if it gives us some insight into improving our model

In [ ]:
#Running the model on the test set and returning the predictions, then converting the raw class score predictions (logits) into numeric labels
test_output = trainer.predict(test_dataset)
test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids #numeric labels of actual test data

#Adding the numeric label, sentiment and a True/False column (comparing actual and preds) to the test df
test_df["pred_label"] = test_preds
test_df["pred_polarity"] = test_df["pred_label"].map(id2label)
test_df["correct"] = test_df["label"] == test_df["pred_label"]

#Printing out accuracy and F1 for the predictions
print(f"\nOverall test accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print(f"Overall test macro-F1: {f1_score(test_labels, test_preds, average='macro'):.4f}")

#Confusion Matrix
labels_sorted = [id2label[i] for i in range(num_labels)]
cm = confusion_matrix(test_labels, test_preds)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
print("\nConfusion matrix (rows=true, cols=predicted):")
print(cm_df)

#Per domain accuracy
print("\nPer-domain accuracy:")
print(test_df.groupby("domain")["correct"].mean())

#Calculating accuracy per aspect counts in a sentence i.e., what is the accuracy for a sentence with 2 aspects vs accuracy for a sentence with 1 aspect?
#Finding mean and count of correct predictions
aspect_counts = (
    test_df.groupby(["domain", "sentence"])["aspect"]
    .transform("count")
    .rename("aspect_count_in_sentence")
)
test_df["aspect_count_in_sentence"] = aspect_counts
print("\nAccuracy by number of aspects in the sentence:")
print(test_df.groupby("aspect_count_in_sentence")["correct"].agg(["mean", "count"]))

# Saving row-level predictions for manual misclassification tagging
test_df.to_csv("absa_test_predictions_1.csv", index=False)
print("\nSaved row-level predictions to absa_test_predictions_1.csv for manual error tagging.")


Overall test accuracy: 0.7770
Overall test macro-F1: 0.7193

Confusion matrix (rows=true, cols=predicted):
          negative  neutral  positive
negative       133       17        17
neutral         27       54        28
positive        13       30       273

Per-domain accuracy:
domain
laptop        0.818966
restaurant    0.750000
Name: correct, dtype: float64

Accuracy by number of aspects in the sentence:
                              mean  count
aspect_count_in_sentence                 
1                         0.792899    507
2                         0.697368     76
3                         0.555556      9

Saved row-level predictions to absa_test_predictions_1.csv for manual error tagging.


Looking at the output, some of the observations are:

*   Negative: 133/167 correct → ~80% recall
*   Neutral: 54/109 correct → ~50% recall
*   Positive: 273/316 correct → ~86% recall

Positive has the highest recall across all sentiments and neutral is clearly weak. This is expected since neutral reviews can be hard to categorize given their borderline nature.

Per domain Accuracy for Laptops is higher - probably because reviews for these could be slightly more technical and nuanced rather than tha casual nature of food-based reviews.

For the accuracy per number of aspects in a sentence, the well-performing segment is the single aspect sentences and it degrades for every addition. But since our test set only has 9 sentences with 3 aspects, we must take this accuracy with a grain of salt.

Overall, the metrics seem pretty good, with our main observations being weaknesses in neutral class and multi-aspect predictions.

In [ ]:
#Let's save the current model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel + tokenizer saved to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model + tokenizer saved to ./absa_deberta_v3_base


# Addressing Neutral Class Imbalance through a Weighted Loss function

Let us see if adding class weights can help in addressing the lower recall for the neutral class

In [ ]:
#Weighted Loss Set-up

from torch.nn import CrossEntropyLoss

#Computing inverse-frequency class weights from the training set.
#Start with raw weights which are reciprocals of each class count, divide each by the total of raw weights just calculated, and multiply by number of classes (3 in our case). These are the final normalized weights
class_counts = train_df["label"].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = class_weights.to(model.device)

print(f"Class weights (by label id {label2id}): {class_weights}")


#Now, augmenting the custom trainer with a weighted loss function using the class weights we have created above
#We feed the weights into the cross entropy loss function, such that if a minority class is incorrectly predicted, the weighted loss is higher than the ordinary loss

class WeightedLossTrainer(Trainer):                                  #original "Trainer" from HF calculates standard classification loss, We create a custom class to customize the trainer to calculate weighted cross-entropy loss
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")                                #popping labels (removing them from input and storing them) so that we don't calculate the default classification loss using the labels
        outputs = model(**inputs)                                    #generates predictions using inputs (without calculating built-in default loss)
        logits = outputs.get("logits")                               #model returns logits

        loss_fct = CrossEntropyLoss(weight=self.class_weights)       #creating our weighted loss function
        loss = loss_fct(logits,labels)                               #logits and labels are used to calculate the weighted loss

        return (loss, outputs) if return_outputs else loss           #providing trainer access to loss as well as model's predictions




Class weights (by label id {'negative': 0, 'neutral': 1, 'positive': 2}): tensor([0.9804, 1.4997, 0.5199], device='cuda:0')


Having created the updated trainer, let us reload a fresh pretrained model before weighted-loss training

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float32,
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

In [ ]:

#Rebuilding using the weighted version instead of plain Trainer
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,           #reusing the same TrainingArguments from before
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights,  #Supplying class weights for our custom loss function
)

trainer.train()

print("\nValidation metrics (best checkpoint, weighted loss):")
print(trainer.evaluate())

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.508632,0.683926,0.758037,0.680463
2,1.161281,0.613295,0.798646,0.737774
3,0.906656,0.544260,0.793570,0.759896


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.508632,0.683926,0.758037,0.680463
2,1.161281,0.613295,0.798646,0.737774
3,0.906656,0.544260,0.793570,0.759896
4,0.847199,0.567869,0.812183,0.766190


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation metrics (best checkpoint, weighted loss):


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.847199,0.567869,4,0.812183,0.766190


{'eval_loss': 0.5678693652153015, 'eval_accuracy': 0.8121827411167513, 'eval_macro_f1': 0.7661895471078825}


In [ ]:
#Same error analysis as we did previously
test_output = trainer.predict(test_dataset)
test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids

test_df["pred_label"] = test_preds
test_df["pred_polarity"] = test_df["pred_label"].map(id2label)
test_df["correct"] = test_df["label"] == test_df["pred_label"]

print(f"\nOverall test accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print(f"Overall test macro-F1: {f1_score(test_labels, test_preds, average='macro'):.4f}")

#Confusion matrix
labels_sorted = [id2label[i] for i in range(num_labels)]
cm = confusion_matrix(test_labels, test_preds)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
print("\nConfusion matrix (rows=true, cols=predicted):")
print(cm_df)

#Per-domain accuracy
print("\nPer-domain accuracy:")
print(test_df.groupby("domain")["correct"].mean())

# Accuracy by aspect-count-per-sentence
aspect_counts = (
    test_df.groupby(["domain", "sentence"])["aspect"]
    .transform("count")
    .rename("aspect_count_in_sentence")
)
test_df["aspect_count_in_sentence"] = aspect_counts
print("\nAccuracy by number of aspects in the sentence:")
print(test_df.groupby("aspect_count_in_sentence")["correct"].agg(["mean", "count"]))

# Save row-level predictions for manual misclassification tagging
test_df.to_csv("absa_test_predictions_2.csv", index=False)
print("\nSaved row-level predictions to absa_test_predictions_2.csv for manual error tagging.")


Overall test accuracy: 0.7770
Overall test macro-F1: 0.7234

Confusion matrix (rows=true, cols=predicted):
          negative  neutral  positive
negative       127       26        14
neutral         24       60        25
positive        11       32       273

Per-domain accuracy:
domain
laptop        0.823276
restaurant    0.747222
Name: correct, dtype: float64

Accuracy by number of aspects in the sentence:
                              mean  count
aspect_count_in_sentence                 
1                         0.788955    507
2                         0.723684     76
3                         0.555556      9

Saved row-level predictions to absa_test_predictions_2.csv for manual error tagging.


The Macro-F1 on the test data has actually fallen compared to our unweighted version.

*   Negative: 127/167 correct → ~76% recall
*   Neutral: 60/109 correct → ~55% recall
*   Positive: 273/316 correct → ~86% recall

Neutral recall has improved slightly, with positive recall remaining same as before, and negative recall worsening. Laptop's sentiment classification has increased marginally

Overall, we can conclude that depending on use-case (whether prioritizing sentiments in the extreme or borderline), we can choose to employ weighted-loss or not, although in this case the model's improvement is pretty negligible honestly, and this performance doesn't provide us a strong argument for its implementation. This lack of improvement could possibly suggest genuine semantic ambiguity in neutral sentiment rather than a fixable class-imbalance problem.

Overall, our ABSA through fine-tuning DeBERTa (without the weighted loss) has provided us decent classification performance. The weighted loss didnt work as intended, proving that a simple fine-tuning for this particular model should do the trick.

